# Forecasting a seven-day adult ICU occupancy breach

For each U.S. state or reporting jurisdiction, can today's hospital report help rank the risk that **at least 90% of adult ICU beds will be occupied on any of the next seven calendar days**? This retrospective study uses the archived HHS state-level reporting snapshot. Its forecast unit is one jurisdiction on one date.

On the held-out period, XGBoost ranked rare breaches better than the current occupancy baseline. The fixed alert threshold still missed most observed breaches. [Data source and dictionary](https://healthdata.gov/d/g62h-syeh)

## Snapshot and eligibility

The checked-in HHS RAW export contains one row per reporting jurisdiction and date. A forecast needs an observed current occupancy and a report on **each** of the next seven dates. A missing future measurement makes the label unknown. Other missing feature values remain missing for XGBoost. Values above 100% remain as reported and may reflect reporting irregularities or over-capacity operation.

In [1]:
from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd
from icu_forecast import (OCCUPANCY, TARGET, benchmark, build_panel, feature_contributions,
                          fit_forecast, plot_contributions, prediction_frame, read_snapshot,
                          split_periods)

raw = read_snapshot()
panel = build_panel(raw)
eligible = panel[[OCCUPANCY, TARGET]].notna().all(axis=1)
display(pd.Series({
    "Source state-days": len(raw),
    "Jurisdictions": panel.state.nunique(),
    "Missing current occupancy": panel[OCCUPANCY].isna().sum(),
    "Current occupancy above 100%": (panel[OCCUPANCY] > 1).sum(),
    "Currently observed with incomplete future week": (panel[OCCUPANCY].notna() & panel[TARGET].isna()).sum(),
    "Eligible origins before boundary purges": eligible.sum(),
}, name="Count").to_frame())

,Count
Source state-days,64703
Jurisdictions,54
Missing current occupancy,7478
Current occupancy above 100%,60
Currently observed with incomplete future week,402
Eligible origins before boundary purges,56823


## Construct calendar-day features and separate evaluation periods

The four original utilization ratios contribute current values, one-day changes, averages over the trailing seven and fourteen calendar dates, trends, and lags of one, three, seven and fourteen dates. Available same-day numerators, denominators and reporting counts also enter the model. State-specific calendars keep missing dates in the correct lag positions. Trailing averages use observed reports in the window including the forecast date.

The July 2022 test cutoff is the original analytical choice. Validation starts April 2022. Training and validation each exclude origins whose outcome windows reach the next partition. The test set covers future dates for participating jurisdictions. Generalization to previously unseen jurisdictions or a future outbreak would require separate evidence.

In [2]:
periods = split_periods(panel)
train, validation, test = (periods[name] for name in ("Train", "Validation", "Test"))
display(pd.DataFrame([{
    "Period": name, "First origin": data.date.min().date(), "Last origin": data.date.max().date(),
    "Jurisdictions": data.state.nunique(), "Origins": len(data), "Breaches": int(data[TARGET].sum()),
    "Breaches (%)": round(100 * data[TARGET].mean(), 2),
} for name, data in periods.items()]).set_index("Period"))

,First origin,Last origin,Jurisdictions,Origins,Breaches,Breaches (%)
Period,,,,,,
Train,2020-07-11,2022-03-24,54,32901,1735,5.27
Validation,2022-04-01,2022-06-23,54,4536,50,1.10
Test,2022-07-01,2023-06-10,54,18630,376,2.02


## Fit the model

Fit the state encoder and numeric feature selection using training rows. Stop boosting when validation log loss stops improving for 50 rounds. Use unweighted binary log loss so class prevalence remains meaningful when scoring forecast probabilities. The random seed is 42.

In [3]:
model = fit_forecast(train, validation)
probabilities = model.probabilities(test)
print(f"Selected boosting rounds: {model.classifier.best_iteration + 1}")

Selected boosting rounds: 174


## Evaluate on dates held out from fitting

Average precision summarizes ranking for a rare event. ROC AUC measures discrimination across thresholds. Brier score measures squared error of the forecast probability, with smaller values indicating better performance. The constant baseline predicts the training event rate. Persistence ranks on today's continuous occupancy and predicts a breach if today's occupancy is already at least 90%. Its Brier score uses that binary forecast. XGBoost alerts at a fixed probability of 0.5.

In [4]:
scores = benchmark(train, test, probabilities)
display(scores.round(4))

,ROC AUC,Average precision,Brier,True alerts,False alerts,Misses
Method,,,,,,
Training prevalence,0.5000,0.0202,0.0208,0,0,376
Occupancy persistence,0.8428,0.3717,0.0178,78,33,298
XGBoost,0.9308,0.3912,0.0154,52,6,324


### Where the fixed alerts missed breaches

An origin counts as a true event if any day in its next week breaches the threshold. Adjacent origins can refer to the same episode, so these counts describe forecast dates. They are not independent hospital events.

In [5]:
review = test[["state", "date", TARGET]].copy()
review["alert"] = probabilities >= 0.5
review["hit"] = review["alert"] & review[TARGET].eq(1)
review["miss"] = ~review["alert"] & review[TARGET].eq(1)
review["false_alert"] = review["alert"] & review[TARGET].eq(0)
by_quarter = review.groupby(review.date.dt.to_period("Q")).agg(
    origins=(TARGET, "size"), breaches=(TARGET, "sum"), correct_alerts=("hit", "sum"),
    missed_breaches=("miss", "sum"), false_alerts=("false_alert", "sum"))
display(by_quarter)

,origins,breaches,correct_alerts,missed_breaches,false_alerts
date,,,,,
2022Q3,4968,98,0,98,0
2022Q4,4968,102,22,80,6
2023Q1,4860,153,30,123,0
2023Q2,3834,23,0,23,0


## Interpret fitted predictions

TreeSHAP values are computed on at most 2,000 seeded test rows using the selected boosting rounds. The seven largest mean absolute contributions describe model sensitivity in log-odds units. Correlated predictors can share credit, and the calculation cannot identify causes of occupancy changes.

In [6]:
importance = feature_contributions(model, test)
figure = plot_contributions(importance)
figure.savefig("SHAP_features.png", dpi=150)
plt.close(figure)
display(importance.head(7).rename("Mean absolute contribution (log-odds)").round(3).to_frame())

,Mean absolute contribution (log-odds)
adult_icu_bed_utilization,1.051
adult_icu_bed_utilization_lag1,0.321
adult_icu_bed_utilization_ma7,0.276
adult_icu_bed_utilization_ma14,0.232
percent_of_inpatients_with_covid_denominator,0.143
adult_icu_bed_utilization_lag3,0.141
adult_icu_bed_covid_utilization_lag14,0.092


![Current adult ICU occupancy contributes most to the fitted score, followed by its one-day lag and seven- and fourteen-day means.](SHAP_features.png)

## Export historical predictions

The CSV preserves the original four columns and contains only eligible held-out forecast origins. The data end in 2023. A model ranking on the last evaluable date describes that historical date.

In [7]:
output = prediction_frame(test, probabilities)
output.to_csv("icu_breach_probabilities.csv", index=False)
print(f"Exported {len(output):,} dated forecasts.")
latest = output.loc[output.date.eq(output.date.max())]
display(latest.nlargest(3, "prob_breach_7d").reset_index(drop=True))

Exported 18,630 dated forecasts.


,state,date,adult_icu_bed_utilization,prob_breach_7d
0,KS,2023-06-10,0.687335,0.083903
1,RI,2023-06-10,0.801325,0.074101
2,DE,2023-06-10,0.733728,0.035891


## Limits on interpretation

The dataset is a historical export of revised aggregate reports. The publication delay and the exact version available to a forecaster on each date are unknown. Changes in reporting coverage, missing outcomes and jurisdiction mix can alter performance. The results establish retrospective predictive performance for this snapshot and period only. This experiment does not establish clinical utility, staffing impact, or a causal contribution of COVID-19 to capacity pressures.